# 🧠 POLYDIM V4.2 - ENTRENAMIENTO EN TINYSHAKESPEARE
Incluye comparación directa contra un Transformer Baseline.

In [3]:
# ============================================================
# POLYDIM V4.2 - ENTRENAMIENTO TINYSHAKESPEARE (CORREGIDO)
# ============================================================
# Fixes: WHT huérfano eliminado, seed fijo, baseline agregado,
#        test PMTP relajado para fase aprendible, comentarios
#        sobre pérdida de información imaginaria en L_q.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import requests
import time
import json
from typing import Tuple, Optional

torch.manual_seed(1337)
np.random.seed(1337)

# -----------------------------------------------------------
# 0. DATASET
# -----------------------------------------------------------
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"[DATA] Longitud: {len(text)}, Vocab: {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# -----------------------------------------------------------
# 1. COMPONENTES POLYDIM
# -----------------------------------------------------------

class MagneticLaplacian(nn.Module):
    """
    Laplaciano Magnético para grafos dirigidos.
    L^(q) = D_s - H^(q)  donde H^(q) = A_s ⊙ exp(i * Theta)
    """
    def __init__(self, N: int, q: float = math.pi / 4.0, skip_k: int = 3):
        super().__init__()
        self.N = N
        self.q = q
        self.skip_k = skip_k

        A = self._build_adjacency(N, skip_k)
        A_s = 0.5 * (A + A.T)
        D_s = np.diag(np.sum(A_s, axis=1))
        Theta = 2 * np.pi * q * (A - A.T)
        H_q = A_s * np.exp(1j * Theta)
        L_q = D_s - H_q

        self.register_buffer('L_q_real', torch.from_numpy(L_q.real).float())
        self.register_buffer('L_q_imag', torch.from_numpy(L_q.imag).float())

    def _build_adjacency(self, N: int, skip_k: int) -> np.ndarray:
        A = np.zeros((N, N), dtype=np.float64)
        for i in range(N - 1):
            A[i, i + 1] = 1.0
        for k in range(2, skip_k + 1):
            for i in range(N - k):
                A[i, i + k] = 1.0
        return A

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T_seq = x.shape[-2]
        L_q = torch.complex(
            self.L_q_real[:T_seq, :T_seq],
            self.L_q_imag[:T_seq, :T_seq]
        )
        if not torch.is_complex(x):
            x = x.to(torch.cfloat)
        y = torch.einsum('ij,...jd->...id', L_q, x)
        return y.real

class FourierEmbedding(nn.Module):
    def __init__(self, D: int, max_len: int = 5000, learnable: bool = True):
        super().__init__()
        self.D = D
        freqs = torch.exp(torch.arange(0, D, 2).float() * (-math.log(10000.0) / D))
        if learnable:
            self.freqs = nn.Parameter(freqs)
        else:
            self.register_buffer('freqs', freqs)

    def forward(self, positions: torch.Tensor) -> torch.Tensor:
        B, N = positions.shape
        angles = positions.unsqueeze(-1) * self.freqs.unsqueeze(0).unsqueeze(0)
        emb = torch.zeros(B, N, self.D, device=positions.device)
        emb[:, :, 0::2] = torch.sin(angles)
        emb[:, :, 1::2] = torch.cos(angles)
        return emb

class IsometricRotation(nn.Module):
    def __init__(self, D: int, use_fft: bool = True):
        super().__init__()
        self.D = D
        self.use_fft = use_fft
        self.phase = nn.Parameter(torch.randn(D) * 0.01)

    def forward(self, x: torch.Tensor, inverse: bool = False) -> torch.Tensor:
        sign = -1 if inverse else 1
        if self.use_fft:
            x = torch.fft.fft(x, dim=-1)
            x = x * torch.exp(1j * sign * self.phase).unsqueeze(0)
            x = torch.fft.ifft(x, dim=-1)
            x = x.real
        return x

class PMTPRouter(nn.Module):
    def __init__(self, D: int, seed: int = 42):
        super().__init__()
        self.D = D
        self.seed = seed
        torch.manual_seed(seed)
        self.rotor = IsometricRotation(D, use_fft=True)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.rotor(x, inverse=False)

    def decode(self, x: torch.Tensor) -> torch.Tensor:
        return self.rotor(x, inverse=True)

class PolydimLayer(nn.Module):
    def __init__(self, D: int, N: int, n_nodes: int, q: float, skip_k: int, dropout: float):
        super().__init__()
        self.D = D
        self.N = N
        self.n_nodes = n_nodes
        self.d_node = D // n_nodes

        self.norm1 = nn.LayerNorm(D)
        self.norm2 = nn.LayerNorm(D)

        self.rotors = nn.ModuleList([
            IsometricRotation(self.d_node, use_fft=True)
            for _ in range(n_nodes)
        ])

        self.laplacian = MagneticLaplacian(N, q=q, skip_k=skip_k)

        self.mlp = nn.Sequential(
            nn.Linear(D, 4 * D),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * D, D),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        residual = x
        x = self.norm1(x)

        x_nodes = x.reshape(B, N, self.n_nodes, self.d_node)
        rotated = []
        for i, rotor in enumerate(self.rotors):
            node = x_nodes[:, :, i, :]
            node_rot = rotor(node)
            rotated.append(node_rot)

        x = torch.stack(rotated, dim=2).reshape(B, N, D)
        x = self.laplacian(x)

        x = residual + x
        residual = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = residual + x
        return x

class PolydimMotorV3(nn.Module):
    def __init__(
        self,
        vocab_size: int = 50000,
        D: int = 4096,
        N: int = 512,
        n_layers: int = 6,
        n_nodes: int = 20,
        q: float = math.pi / 4.0,
        skip_k: int = 3,
        dropout: float = 0.1
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.D = D
        self.N = N
        self.n_nodes = n_nodes
        self.d_node = D // n_nodes

        assert D % n_nodes == 0
        assert (self.d_node & (self.d_node - 1)) == 0

        self.token_embed = nn.Embedding(vocab_size, D)
        self.pos_embed = FourierEmbedding(D, max_len=N)
        self.dropout = nn.Dropout(dropout)
        self.to_manifold = nn.Linear(D, D)

        self.layers = nn.ModuleList([
            PolydimLayer(D, N, n_nodes, q, skip_k, dropout)
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(D)
        self.to_vocab = nn.Linear(D, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, return_latent: bool = False) -> torch.Tensor:
        B, N = input_ids.shape
        token_emb = self.token_embed(input_ids)
        pos_ids = torch.arange(N, device=input_ids.device).unsqueeze(0).expand(B, -1)
        pos_emb = self.pos_embed(pos_ids)
        x = self.dropout(token_emb + pos_emb)
        x = self.to_manifold(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        if return_latent: return x
        return self.to_vocab(x)

# -----------------------------------------------------------
# 2. BASELINE TRANSFORMER
# -----------------------------------------------------------

class BaselineTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        D: int = 256,
        N: int = 128,
        n_layers: int = 4,
        n_heads: int = 4,
        dropout: float = 0.1
    ):
        super().__init__()
        self.D = D
        self.N = N

        self.token_embed = nn.Embedding(vocab_size, D)
        self.pos_embed = nn.Embedding(N, D)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D,
            nhead=n_heads,
            dim_feedforward=4*D,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.norm = nn.LayerNorm(D)
        self.to_vocab = nn.Linear(D, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, N = input_ids.shape
        token_emb = self.token_embed(input_ids)
        pos_ids = torch.arange(N, device=input_ids.device).unsqueeze(0).expand(B, -1)
        pos_emb = self.pos_embed(pos_ids)
        x = self.dropout(token_emb + pos_emb)

        mask = nn.Transformer.generate_square_subsequent_mask(N, device=input_ids.device)
        x = self.transformer(x, mask=mask, is_causal=True)

        x = self.norm(x)
        return self.to_vocab(x)

# -----------------------------------------------------------
# 3. CONFIGURACIÓN ENTRENAMIENTO
# -----------------------------------------------------------

CONFIG = {
    "model": "PolydimMotorV3",
    "vocab_size": vocab_size,
    "D": 256,
    "N": 128,
    "n_layers": 4,
    "n_nodes": 4,
    "q": math.pi / 4.0,
    "skip_k": 3,
    "dropout": 0.1,
    "batch_size": 32,
    "block_size": 128,
    "max_iters": 5000,
    "eval_interval": 500,
    "learning_rate": 1e-3,
    "weight_decay": 0.01,
    "device": 'cuda' if torch.cuda.is_available() else 'cpu',
    "seed": 1337
}

print(f"\n[CONFIG] Entrenando en: {CONFIG['device']}")

torch.manual_seed(CONFIG['seed'])

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - CONFIG['block_size'], (CONFIG['batch_size'],))
    x = torch.stack([data[i:i+CONFIG['block_size']] for i in ix])
    y = torch.stack([data[i+1:i+CONFIG['block_size']+1] for i in ix])
    return x.to(CONFIG['device']), y.to(CONFIG['device'])

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(100)
        accs = torch.zeros(100)
        for k in range(100):
            X, Y = get_batch(split)
            logits = model(X)
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            Y_flat = Y.view(B*T)
            loss = F.cross_entropy(logits_flat, Y_flat)
            losses[k] = loss.item()
            preds = logits_flat.argmax(dim=-1)
            accs[k] = (preds == Y_flat).float().mean().item()
        out[split] = {'loss': losses.mean().item(), 'acc': accs.mean().item()}
    model.train()
    return out

# -----------------------------------------------------------
# 4. ENTRENAMIENTO POLYDIM
# -----------------------------------------------------------

print("\n🚀 INICIANDO ENTRENAMIENTO POLYDIM...")

model_polydim = PolydimMotorV3(
    vocab_size=CONFIG['vocab_size'], D=CONFIG['D'], N=CONFIG['N'],
    n_layers=CONFIG['n_layers'], n_nodes=CONFIG['n_nodes'],
    q=CONFIG['q'], skip_k=CONFIG['skip_k'], dropout=CONFIG['dropout']
).to(CONFIG['device'])

optimizer = torch.optim.AdamW(model_polydim.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

history = {'polydim': {'iters': [], 'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}}

t0 = time.time()
for iter in range(CONFIG['max_iters'] + 1):
    if iter % CONFIG['eval_interval'] == 0:
        losses = estimate_loss(model_polydim)
        elapsed = time.time() - t0
        print(f"Step {iter:5d} | train_loss {losses['train']['loss']:.4f} | val_loss {losses['val']['loss']:.4f} | train_acc {losses['train']['acc']:.4f} | val_acc {losses['val']['acc']:.4f} | time {elapsed:.1f}s")
        history['polydim']['iters'].append(iter)
        history['polydim']['train_loss'].append(losses['train']['loss'])
        history['polydim']['val_loss'].append(losses['val']['loss'])
        history['polydim']['train_acc'].append(losses['train']['acc'])
        history['polydim']['val_acc'].append(losses['val']['acc'])

    xb, yb = get_batch('train')
    logits = model_polydim(xb)
    loss = F.cross_entropy(logits.view(-1, CONFIG['vocab_size']), yb.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("✅ Entrenamiento Polydim finalizado.")

# -----------------------------------------------------------
# 5. ENTRENAMIENTO BASELINE TRANSFORMER
# -----------------------------------------------------------

print("\n🚀 INICIANDO ENTRENAMIENTO BASELINE TRANSFORMER...")

model_baseline = BaselineTransformer(
    vocab_size=CONFIG['vocab_size'], D=CONFIG['D'], N=CONFIG['N'],
    n_layers=CONFIG['n_layers'], n_heads=4, dropout=CONFIG['dropout']
).to(CONFIG['device'])

optimizer_base = torch.optim.AdamW(model_baseline.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

history['baseline'] = {'iters': [], 'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

t0 = time.time()
for iter in range(CONFIG['max_iters'] + 1):
    if iter % CONFIG['eval_interval'] == 0:
        losses = estimate_loss(model_baseline)
        elapsed = time.time() - t0
        print(f"Step {iter:5d} | train_loss {losses['train']['loss']:.4f} | val_loss {losses['val']['loss']:.4f} | train_acc {losses['train']['acc']:.4f} | val_acc {losses['val']['acc']:.4f} | time {elapsed:.1f}s")
        history['baseline']['iters'].append(iter)
        history['baseline']['train_loss'].append(losses['train']['loss'])
        history['baseline']['val_loss'].append(losses['val']['loss'])
        history['baseline']['train_acc'].append(losses['train']['acc'])
        history['baseline']['val_acc'].append(losses['val']['acc'])

    xb, yb = get_batch('train')
    logits = model_baseline(xb)
    loss = F.cross_entropy(logits.view(-1, CONFIG['vocab_size']), yb.view(-1))

    optimizer_base.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_base.step()

print("✅ Entrenamiento Baseline finalizado.")

# -----------------------------------------------------------
# 6. GENERAR TEXTO Y GUARDAR LOGS
# -----------------------------------------------------------

with open('training_logs.json', 'w') as f:
    json.dump(history, f, indent=2)

print("\n📊 Logs guardados localmente en training_logs.json")

print("\n🧠 GENERANDO TEXTO (Polydim)...")
context = torch.zeros((1, 1), dtype=torch.long, device=CONFIG['device'])
model_polydim.eval()
generated = []
for _ in range(500):
    cond = context[:, -CONFIG['block_size']:]
    logits = model_polydim(cond)
    probs = F.softmax(logits[:, -1, :], dim=-1)
    next_tok = torch.multinomial(probs, num_samples=1)
    context = torch.cat((context, next_tok), dim=1)
    generated.append(next_tok.item())
print(decode(generated))

print("\n🧠 GENERANDO TEXTO (Baseline)...")
context = torch.zeros((1, 1), dtype=torch.long, device=CONFIG['device'])
model_baseline.eval()
generated = []
for _ in range(500):
    cond = context[:, -CONFIG['block_size']:]
    logits = model_baseline(cond)
    probs = F.softmax(logits[:, -1, :], dim=-1)
    next_tok = torch.multinomial(probs, num_samples=1)
    context = torch.cat((context, next_tok), dim=1)
    generated.append(next_tok.item())
print(decode(generated))



[DATA] Longitud: 1115394, Vocab: 65

[CONFIG] Entrenando en: cuda

🚀 INICIANDO ENTRENAMIENTO POLYDIM...
Step     0 | train_loss 4.2437 | val_loss 4.2399 | train_acc 0.0098 | val_acc 0.0098 | time 2.0s
Step   500 | train_loss 1.7750 | val_loss 1.7953 | train_acc 0.4699 | val_acc 0.4706 | time 17.5s
Step  1000 | train_loss 1.2703 | val_loss 1.3286 | train_acc 0.5854 | val_acc 0.5693 | time 33.3s
Step  1500 | train_loss 1.0284 | val_loss 1.0991 | train_acc 0.6477 | val_acc 0.6295 | time 49.3s
Step  2000 | train_loss 0.8975 | val_loss 0.9908 | train_acc 0.6837 | val_acc 0.6608 | time 65.3s
Step  2500 | train_loss 0.8200 | val_loss 0.9270 | train_acc 0.7121 | val_acc 0.6802 | time 81.5s
Step  3000 | train_loss 0.7477 | val_loss 0.8689 | train_acc 0.7326 | val_acc 0.7024 | time 98.0s
Step  3500 | train_loss 0.7176 | val_loss 0.8244 | train_acc 0.7419 | val_acc 0.7125 | time 114.2s
Step  4000 | train_loss 0.6669 | val_loss 0.7837 | train_acc 0.7599 | val_acc 0.7285 | time 130.6s
Step  4500 | 

/tmp/ipykernel_523/3830943544.py:271: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


Step     0 | train_loss 4.1739 | val_loss 4.1771 | train_acc 0.0634 | val_acc 0.0616 | time 2.4s
Step   500 | train_loss 2.0768 | val_loss 2.1296 | train_acc 0.3842 | val_acc 0.3654 | time 24.0s
Step  1000 | train_loss 1.6479 | val_loss 1.8058 | train_acc 0.4980 | val_acc 0.4581 | time 45.6s
Step  1500 | train_loss 1.5028 | val_loss 1.6895 | train_acc 0.5386 | val_acc 0.4961 | time 67.0s
Step  2000 | train_loss 1.4208 | val_loss 1.6289 | train_acc 0.5603 | val_acc 0.5165 | time 88.5s
Step  2500 | train_loss 1.3837 | val_loss 1.5884 | train_acc 0.5709 | val_acc 0.5275 | time 110.0s
Step  3000 | train_loss 1.3494 | val_loss 1.5760 | train_acc 0.5800 | val_acc 0.5322 | time 131.7s
Step  3500 | train_loss 1.3291 | val_loss 1.5559 | train_acc 0.5854 | val_acc 0.5369 | time 153.5s
Step  4000 | train_loss 1.3038 | val_loss 1.5304 | train_acc 0.5917 | val_acc 0.5458 | time 175.1s
Step  4500 | train_loss 1.2836 | val_loss 1.5275 | train_acc 0.5974 | val_acc 0.5461 | time 196.9s
Step  5000 | tra